In [7]:
import sys
import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

# Add current directory to path just in case VS Code environment is lost
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

print("Step 1: Ingesting standard libraries... Done.")

print("Step 2: Loading custom engines (this can lag due to shap/lightgbm initialization)...")
from src.data_engine import SectorDataEngine
print("-> Data Engine imported successfully.")

from src.model_engine import SectorModelEngine
print("-> Model Engine imported successfully.")

print("\nStep 3: Generating synthetic multi-sector stock data...")
np.random.seed(42)
sectors = {
    'Banking': ['SBI', 'HDFC', 'ICICI'],
    'IT': ['TCS', 'INFY', 'WIPRO'],
    'Pharma': ['SUN', 'REDDY', 'CIPLA', 'LUPIN']
}

dates = [datetime(2026, 1, 1) + timedelta(days=i) for i in range(100)]
data_rows = []

for date in dates:
    if date.weekday() >= 5:
        continue
    for sector, tickers in sectors.items():
        for ticker in tickers:
            base_price = 500 if sector == 'Banking' else (1500 if sector == 'IT' else 800)
            random_return = np.random.normal(0.0005, 0.015) 
            close_price = base_price * (1 + random_return)
            
            data_rows.append({
                'date': date.strftime('%Y-%m-%d'),
                'ticker': ticker,
                'sector': sector,
                'close': close_price
            })

raw_market_data = pd.DataFrame(data_rows)
print(f"-> Dataset successfully created with shape: {raw_market_data.shape}")


# --- Simulate Macro-Fundamental stream (Repo Rate & Currency Pairs) ---
print("Generating tracking macro-fundamental database...")
macro_dates = [datetime(2026, 1, 1) + timedelta(days=i) for i in range(100)]
macro_rows = []

current_rate = 6.50
current_fx = 83.50

for date in macro_dates:
    if date.weekday() >= 5:
        continue
    # Simulate a mid-timeline interest rate cut by the RBI
    if date.strftime('%Y-%m-%d') == '2026-02-15':
        current_rate = 6.25 
        
    # Introduce random walk volatility for the currency exchange rate
    current_fx += np.random.normal(0.01, 0.10)
    
    macro_rows.append({
        'date': date.strftime('%Y-%m-%d'),
        'repo_rate': current_rate,
        'usd_inr': current_fx
    })

macro_dataframe = pd.DataFrame(macro_rows)
print(f"-> Macro database successfully created with shape: {macro_dataframe.shape}")

Step 1: Ingesting standard libraries... Done.
Step 2: Loading custom engines (this can lag due to shap/lightgbm initialization)...
-> Data Engine imported successfully.
-> Model Engine imported successfully.

Step 3: Generating synthetic multi-sector stock data...
-> Dataset successfully created with shape: (720, 4)
Generating tracking macro-fundamental database...
-> Macro database successfully created with shape: (72, 3)


In [8]:
# Initialize data engine
data_engine = SectorDataEngine()

# We will store processed datasets per sector here
processed_sector_data = {}
all_sectors = raw_market_data['sector'].unique()

print("Processing indicators with macro overlays sector by sector...")
for sector in all_sectors:
    # Passing both datasets down into our upgraded engine method
    sector_df = data_engine.get_sector_data(raw_market_data, macro_dataframe, sector)
    processed_sector_data[sector] = sector_df
    print(f"-> {sector} sector processed. Shape: {sector_df.shape} | Memory optimized.")

Processing indicators with macro overlays sector by sector...


TypeError: SectorDataEngine.get_sector_data() takes 3 positional arguments but 4 were given

In [ ]:
# Define features and optimization target matching our pipeline
# Add your macro parameters into the active feature array
features = ['close', 'sma_10', 'sma_50', 'rsi_14', 'repo_rate', 'usd_inr', 'usd_inr_momentum']
target = 'target_return'

print("\nBeginning targeted model training and insight extraction...")

for sector, df in processed_sector_data.items():
    print(f"\n==================== SECTOR: {sector.upper()} ====================")
    
    # Chronological train/test split to prevent data leakage
    df = df.sort_values('date').reset_index(drop=True)
    split_idx = int(len(df) * 0.8)
    
    train_df = df.iloc[:split_idx]
    val_df = df.iloc[split_idx:]
    
    # Initialize and train our single-node LightGBM and XGBoost module
    model_engine = SectorModelEngine(features=features, target=target)

    # Updated method name to reflect both models training
    model_engine.train_sector_models(train_df, val_df)
    
    # Extract the absolute latest market snapshot (today's close) to simulate tomorrow's forecast
    latest_snapshot = df.groupby('ticker').last().reset_index()
    
    # Updated method name for the blended predictions
    advisor_dashboard = model_engine.generate_ensemble_insights(latest_snapshot)
    
    print(f"\nGenerated Next-Day Insights for {sector} Advisors:")
    # Format return as a percentage for clear readability
    advisor_dashboard['predicted_next_day_return'] = (advisor_dashboard['predicted_next_day_return'] * 100).round(2).astype(str) + '%'
    print(advisor_dashboard[['ticker', 'close', 'predicted_next_day_return', 'primary_driver', 'secondary_driver']])


Beginning targeted model training and insight extraction...

==================== SECTOR: BANKING ====================

Generated Next-Day Insights for Banking Advisors:
  ticker       close predicted_next_day_return primary_driver secondary_driver
0   HDFC  508.117584                    -0.89%          close           sma_10
1  ICICI  494.967407                     1.42%          close           sma_10
2    SBI  496.329590                     1.02%          close           sma_10

==================== SECTOR: IT ====================

Generated Next-Day Insights for IT Advisors:
  ticker        close predicted_next_day_return primary_driver  \
0   INFY  1465.725830                     1.99%          close   
1    TCS  1469.059570                     2.03%          close   
2  WIPRO  1514.385254                    -1.26%          close   

  secondary_driver  
0           rsi_14  
1           sma_10  
2           rsi_14  

==================== SECTOR: PHARMA ====================

Gener